# Seleção de candidatos a regex com base nos músculos

## 1. Obter os exercícios

In [67]:
import re
import pandas as pd
from api import get_list

dados = get_list("exerciseinfo/", {"language__code": "en", "limit": 100})

linhas = []
for ex in dados:
    traducao = next((t for t in ex.get("translations", []) if t.get("language") == 2), {})
    nome = traducao.get("name", "") or ""
    linhas.append({
        "ID": ex["id"],
        "Nome": nome,
        "Categoria": (ex.get("category") or {}).get("name", ""),
        "Músculos": len(ex.get("muscles") or []) + len(ex.get("muscles_secondary") or []),
        "Palavras": set(re.findall(r"[^\W\d_]+", nome.lower()))
    })

df = pd.DataFrame(linhas)
print("Total de exercícios:", len(df))
display(df[["ID", "Nome", "Categoria", "Músculos"]].head(8))

Total de exercícios: 954


,ID,Nome,Categoria,Músculos
0,9,2 Handed Kettlebell Swing,Abs,4
1,12,Seated Hip Adduction,Legs,1
2,20,Arnold Shoulder Press,Shoulders,3
3,31,Axe Hold,Arms,0
4,41,Barbell Ab Rollout,Abs,1
5,43,Barbell Hack Squats,Legs,3
6,46,Barbell Lunges Standing,Legs,2
7,48,Barbell Reverse Wrist Curl,Arms,0


## 2. Selecionar as palavras a analisar

In [68]:
df["Palavras"] = df["Nome"].str.lower().apply(lambda nome: re.findall(r"[a-z]+", nome))

palavras = df.explode("Palavras")

ignorar = {
    # Artigos, preposições e conectores
    "a", "and", "at", "between", "for", "in",
    "into", "it", "no", "of", "off", "on",
    "or", "the", "through", "to", "using",
    "while", "with", "around", "behind", "away",

    # Quantidades e modificadores genéricos
    "one", "two", "three", "four", "single",
    "double", "half", "full", "multi",

    # Adjetivos e advérbios pouco informativos
    "active", "alternative", "basic", "continuous",
    "controlled", "easy", "free", "gentle",
    "guided", "heavy", "modified", "negative",
    "slow", "standard", "slight", "small",
    "extreme", "loaded", "long", "brisk",

    # Posições e características genéricas
    "standing", "seated", "sitting", "lying",
    "laying", "left", "right", "forward",
    "backward", "middle", "mid", "horizontal",
    "upright", "diagonal", "flat",

    # Palavras genéricas e administrativas
    "custom", "exercise", "exercises",
    "workout", "workouts", "session", "sets",
    "training", "variation", "count", "width",
    "failure", "timed", "paused", "pause", "zone",

    # Conectores e modificadores noutras línguas
    "al", "alle", "con", "de", "mit",
    "alto", "alta", "ancho", "estrecho",
    "lento", "avanti", "seduto", "inclinado",

    # Siglas sem significado claro para a classificação
    "db", "bb", "sz", "ez", "hd", "nb",
    "mp", "ol", "ss", "ir", "mm", "jm",
    "l", "m", "n", "s", "t", "v",
    "w", "y", "d", "i", "ytw", "ytwl",
    "ywts"

    # Fragmentos de palavras e cores
    "eca", "mu", "jal",
    "yellow", "green", "black"
}

palavras = palavras[~palavras["Palavras"].isin(ignorar)]

frequencias = (palavras.groupby("Palavras", as_index=False)
              .agg(Exercícios=("ID", "nunique"), Exemplo=("Nome", "first"))
              .sort_values(["Exercícios", "Palavras"], ascending=[False, True]))

display(frequencias)

,Palavras,Exercícios,Exemplo
383,press,76,Arnold Shoulder Press
80,cable,72,Biceps Curl With Cable
167,dumbbell,72,Bent Over Dumbbell Rows (Single Arm)
516,stretch,60,Left neck stretch
300,leg,49,Single Leg Extension
...,...,...,...
587,windshield,1,Windshield Wipers
590,woodchoppers,1,Cable Woodchoppers
592,yoga,1,Yoga exercise: Cow-cat
593,ywts,1,YWTs


In [69]:
palavras = frequencias["Palavras"].dropna().str.lower().unique()
print("Palavras para analisar:", len(palavras))

Palavras para analisar: 595


## 3. Calcular as proporções por palavra

In [70]:
validos = df[(df["Categoria"].str.lower() != "cardio") & (df["Músculos"] > 0)]

resultados = []
for termo in palavras:
    encontrados = validos[validos["Palavras"].apply(lambda p: termo in p)]
    total = len(encontrados)
    if total == 0:
        continue

    apenas_1 = int((encontrados["Músculos"] == 1).sum())
    mais_que_1 = total - apenas_1

    resultados.append({
        "Termo": termo,
        "Exercícios": total,
        "1 músculo": apenas_1,
        "2+ músculos": mais_que_1,
        "% 1": round(100 * apenas_1 / total, 1),
        "% 2+": round(100 * mais_que_1 / total, 1),
        "Exemplo real": encontrados.iloc[0]["Nome"],
        "ID": encontrados.iloc[0]["ID"]
    })

evidencias = pd.DataFrame(resultados).sort_values("Exercícios", ascending=False)
print("Exercícios com dados válidos:", len(validos))
print("1 músculo:", int((validos["Músculos"] == 1).sum()))
print("2+ músculos:", int((validos["Músculos"] >= 2).sum()))
display(evidencias.head(35))

Exercícios com dados válidos: 770
1 músculo: 276
2+ músculos: 494


,Termo,Exercícios,1 músculo,2+ músculos,% 1,% 2+,Exemplo real,ID
0,press,74,23,51,31.1,68.9,Arnold Shoulder Press,20
1,cable,67,44,23,65.7,34.3,Biceps Curl With Cable,95
2,dumbbell,67,23,44,34.3,65.7,Bent Over Dumbbell Rows (Single Arm),81
4,leg,46,20,26,43.5,56.5,Single Leg Extension,71
6,pull,39,6,33,15.4,84.6,Close-grip Lat Pull Down,158
5,curl,38,21,17,55.3,44.7,Barbell Wrist Curl,51
8,squat,37,2,35,5.4,94.6,Braced Squat,124
7,row,36,11,25,30.6,69.4,Incline Dumbbell Row,310
3,stretch,34,14,20,41.2,58.8,Elevated prayer stretch,1027
9,machine,33,15,18,45.5,54.5,Calf Press Using Leg Press Machine,146


## 4. Aplicar critérios mínimos e sugerir candidatos

Definimos **pelo menos 5 exercícios por palavra** (para não decidir com base em apenas 1 ou 2 ocorrências) e **60% de concordância** com uma das duas faixas musculares. São parâmetros **exploratórios**, não limiares cientificamente validados; podem ser comparados com outros valores.

- `isolated`: 60% ou mais das ocorrências válidas com 1 músculo.
- `compound`: 60% ou mais das ocorrências válidas com 2+ músculos.
- **Não selecionado:** palavras raras ou sem predominância clara. **Não as colocar no `OTHER_PATTERN`**: falta de evidência não implica exercício do tipo *other*.

Os candidatos a `cardio` e a `other` **não podem ser derivados deste critério muscular**. Mantêm a lógica anterior: categoria Cardio e padrões próprios previamente analisados.

In [71]:
MINIMO = 6
LIMIAR = 66.67

evidencias["Sugestão"] = "não selecionar"
evidencias.loc[
    (evidencias["Exercícios"] >= MINIMO) & (evidencias["% 1"] >= LIMIAR),
    "Sugestão"
] = "isolated"
evidencias.loc[
    (evidencias["Exercícios"] >= MINIMO) & (evidencias["% 2+"] >= LIMIAR),
    "Sugestão"
] = "compound"

display(evidencias[["Termo", "Exercícios", "% 1", "% 2+", "Sugestão", "Exemplo real"]].head(40))

candidatos = {
    "compound": evidencias.loc[evidencias["Sugestão"] == "compound", "Termo"].tolist(),
    "isolated": evidencias.loc[evidencias["Sugestão"] == "isolated", "Termo"].tolist()
}
print("Candidatos compound:", candidatos["compound"])
print("Candidatos isolated:", candidatos["isolated"])

,Termo,Exercícios,% 1,% 2+,Sugestão,Exemplo real
0,press,74,31.1,68.9,compound,Arnold Shoulder Press
1,cable,67,65.7,34.3,não selecionar,Biceps Curl With Cable
2,dumbbell,67,34.3,65.7,não selecionar,Bent Over Dumbbell Rows (Single Arm)
4,leg,46,43.5,56.5,não selecionar,Single Leg Extension
6,pull,39,15.4,84.6,compound,Close-grip Lat Pull Down
5,curl,38,55.3,44.7,não selecionar,Barbell Wrist Curl
8,squat,37,5.4,94.6,compound,Braced Squat
7,row,36,30.6,69.4,compound,Incline Dumbbell Row
3,stretch,34,41.2,58.8,não selecionar,Elevated prayer stretch
9,machine,33,45.5,54.5,não selecionar,Calf Press Using Leg Press Machine


Candidatos compound: ['press', 'pull', 'squat', 'row', 'ups', 'up', 'bench', 'incline', 'push', 'front', 'squats', 'plank', 'calf', 'biceps', 'deadlift', 'kettlebell', 'down', 'sit', 'hold', 'trx', 'wide', 'dips', 'wall', 'knee', 'lunges', 'floor', 'split', 'bridge', 'clean', 'thrust', 'rowing', 'narrow', 'smith']
Candidatos isolated: ['curls', 'extension', 'triceps', 'lateral', 'fly', 'dumbbells', 'overhead', 'tricep', 'pushdown', 'preacher', 'external']


In [72]:
padroes = {}

for tipo, termos in candidatos.items():
    if termos:
        palavras = [re.escape(termo) for termo in termos]
        padroes[tipo] = r"\b(?:" + "|".join(palavras) + r")\b"
    else:
        padroes[tipo] = None

    print(f"{tipo.upper()}_PATTERN = {padroes[tipo]!r}")

COMPOUND_PATTERN = '\\b(?:press|pull|squat|row|ups|up|bench|incline|push|front|squats|plank|calf|biceps|deadlift|kettlebell|down|sit|hold|trx|wide|dips|wall|knee|lunges|floor|split|bridge|clean|thrust|rowing|narrow|smith)\\b'
ISOLATED_PATTERN = '\\b(?:curls|extension|triceps|lateral|fly|dumbbells|overhead|tricep|pushdown|preacher|external)\\b'
